# Fine-tuning a Model and Building an Image Classification App

In [ ]:
!pip install tensorflow==2.16.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 589.8/589.8 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 50.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.18.0
    Uninstalling tensorboard-2.18.0:
      Successfully uninstalled tensorboard-2.18.0
  Attempting uninstall: ml-dtypes
    Fou

In [ ]:
!python --version

Python 3.11.13


In [ ]:
!unzip /content/converted_keras.zip -d /content/data

unzip:  cannot find or open /content/converted_keras.zip, /content/converted_keras.zip.zip or /content/converted_keras.zip.ZIP.


In [ ]:
!ls /content/data

ls: cannot access '/content/data': No such file or directory


In [ ]:
!pip install --upgrade ml-dtypes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 27.2 MB/s eta 0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalling ml-dtypes-0.3.2:
      Successfully uninstalled ml-dtypes-0.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.16.1 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.3 which is incompatible.
tensorflow-decision-forests 1.11.0 requires tensorflow==2.18.0, but you have tensorflow 2.16.1 which is incompatible.
tensorflow-text 2.18.1 requires tensorflow<2.19,>=2.18.0, but you have tensorflow 2.16.1 which is incompatible.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.16.1 which is incompatible.


In [ ]:
!pip show ml-dtypes

Name: ml_dtypes
Version: 0.5.3
Summary: ml_dtypes is a stand-alone implementation of several NumPy dtype extensions used in machine learning.
Home-page: https://github.com/jax-ml/ml_dtypes
Author: 
Author-email: ml_dtypes authors <ml_dtypes@google.com>
License: 
Location: /usr/local/lib/python3.11/dist-packages
Requires: numpy
Required-by: jax, jaxlib, keras, tensorflow, tensorstore


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import DepthwiseConv2D
from tensorflow.keras.utils import custom_object_scope
import numpy as np
from PIL import Image, ImageOps

print("Версия TensorFlow:", tf.__version__)

def custom_depthwise_conv2d(*args, **kwargs):
    kwargs.pop('groups', None)
    return DepthwiseConv2D(*args, **kwargs)

def simple_depthwise_conv2d(config):
    config.pop('groups', None)
    config.pop('name', None)
    return DepthwiseConv2D.from_config(config)

class_names = open("/content/labels.txt", "r").readlines()
print("📋 Классы модели:", [name.strip() for name in class_names])

AttributeError: `np.complex_` was removed in the NumPy 2.0 release. Use `np.complex128` instead.

In [ ]:
import os
print("Файлы в Colab:")
for file in os.listdir('/content/'):
    if 'd17e8542d0a1b2b6ca92eac0a29a9e25' in file:
        print(f"Найден файл: {file}")

# Evaluation

In [ ]:
import os
from keras.models import load_model
from keras.utils import custom_object_scope
from keras.layers import DepthwiseConv2D
from PIL import Image, ImageOps
import numpy as np
import tensorflow as tf

# Disable scientific notation for clarity
np.set_printoptions(suppress=True)

if not os.path.exists("/content/keras_model.h5"):
    print("Модель не найдена")
    print("Пожалуйста:")
    print("1. Загрузите ваш ZIP файл модели через панель файлов слева")
    print("2. Выполните команду распаковки:")
    print("   !unzip /content/ваш_файл.zip -d /content/")
else:
    print("Файлы модели найдены")
    print("Содержимое /content/:")
    print(os.listdir("/content/"))

In [ ]:
def custom_depthwise_conv2d(*args, **kwargs):
    kwargs.pop('groups', None)
    return DepthwiseConv2D(*args, **kwargs)

print("Загружаем модель...")

try:
    with custom_object_scope({'DepthwiseConv2D': custom_depthwise_conv2d}):
        model = load_model("/content/keras_model.h5", compile=False)
    print("Модель успешно загружена!")

    class_names = open("/content/labels.txt", "r").readlines()
    print("Классы модели:", [name.strip() for name in class_names])

except Exception as e:
    print(f"Ошибка загрузки модели: {e}")
    print("Нужно загрузить модель заново")

In [ ]:
import numpy as np
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

def predict_image(image_path, model, class_names):
    """Функция для предсказания класса изображения"""
    data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)

    image = Image.open(image_path).convert("RGB")

    original_image = image.copy()

    size = (224, 224)
    image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)

    image_array = np.asarray(image)
    normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1

    data[0] = normalized_image_array

    prediction = model.predict(data)
    index = np.argmax(prediction)
    confidence_score = prediction[0][index]

    return original_image, index, confidence_score, prediction[0]

In [ ]:
clean_class_names = ['Кошка', 'Собака']

print("Переименованные классы:", clean_class_names)

test_image_path = "/content/d17e8542d0a1b2b6ca92eac0a29a9e25.jpg"

try:
    original, index, confidence, all_predictions = predict_image(test_image_path, model, class_names)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(original)
    plt.title("Исходное изображение")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    colors = ['lightblue' if i != index else 'lightcoral' for i in range(len(clean_class_names))]

    bars = plt.bar(clean_class_names, all_predictions, color=colors)
    plt.title("Вероятности классов")
    plt.ylabel("Вероятность")
    plt.xticks(rotation=45)

    for bar, value in zip(bars, all_predictions):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{value:.2%}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

    print(f"\nРЕЗУЛЬТАТ:")
    print(f"Класс: {clean_class_names[index]}")
    print(f"Уверенность: {confidence:.2%}")

    print(f"\nВсе вероятности:")
    for i, (clean_name, prob) in enumerate(zip(clean_class_names, all_predictions)):
        marker = "" if i == index else "  "
        print(f"{marker} {clean_name}: {prob:.2%}")

except FileNotFoundError:
    print("Файл не найден!")

In [ ]:
test_image_path = "/content/собака.jpg"

try:
    original, index, confidence, all_predictions = predict_image(test_image_path, model, class_names)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(original)
    plt.title("Исходное изображение")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    colors = ['lightblue' if i != index else 'lightcoral' for i in range(len(clean_class_names))]

    bars = plt.bar(clean_class_names, all_predictions, color=colors)
    plt.title("Вероятности классов")
    plt.ylabel("Вероятность")
    plt.xticks(rotation=45)

    for bar, value in zip(bars, all_predictions):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{value:.2%}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

    print(f"\nРЕЗУЛЬТАТ:")
    print(f"Класс: {clean_class_names[index]}")
    print(f"Уверенность: {confidence:.2%}")


    print(f"\nВсе вероятности:")
    for i, (clean_name, prob) in enumerate(zip(clean_class_names, all_predictions)):
        marker = "" if i == index else "  "
        print(f"{marker} {clean_name}: {prob:.2%}")

except FileNotFoundError:
    print("Файл не найден.")


# Testing

In [ ]:
!pip install opendatasets

In [ ]:
import opendatasets

In [ ]:
import opendatasets as od

dataset_url = 'https://www.kaggle.com/datasets/tongpython/cat-and-dog?resource=download'
od.download(dataset_url)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = "/content/cat-and-dog/training_set/training_set"
test_dir = "/content/cat-and-dog/test_set/test_set"

class_names = ["cats", "dogs"]

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    classes=class_names,
    subset='training',
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=True
)

test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    classes=class_names,
    shuffle=False
)

print(train_gen.class_indices)

In [ ]:
# Скомпилируем модель для оценки
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# Оценим модель на тестовых данных

import math
steps = math.ceil(test_gen.samples / test_gen.batch_size)

test_loss, test_accuracy = model.evaluate(test_gen, steps=steps)
print(f"Текущая точность на тестовых данных: {test_accuracy:.2%}")

# Статистика
print(f"\nСтатистика датасета:")
print(f"Тренировочные изображения: {train_gen.samples}")
print(f"Валидационные изображения: {val_gen.samples}")
print(f"Тестовые изображения: {test_gen.samples}")
print(f"Всего изображений: {train_gen.samples + val_gen.samples + test_gen.samples}")

In [ ]:
x_test, y_test = [], []
for i in range(len(test_gen)):
    x_batch, y_batch = test_gen[i]
    x_test.append(x_batch)
    y_test.append(y_batch)
    if i >= len(test_gen) - 1:
        break

X_test = np.vstack(x_test)
Y_test = np.vstack(y_test)

In [ ]:
Y_pred = model.predict(test_gen)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

Y_pred_classes = Y_pred.argmax(1)
Y_true_classes = test_gen.classes


true_labels = [class_names[i] for i in Y_true_classes]
pred_labels = [class_names[i] for i in Y_pred_classes]

# Считаем confucion matrix
confusion_mtx = confusion_matrix(true_labels, pred_labels)

# Строим confucion matrix
disp = ConfusionMatrixDisplay(confusion_mtx, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues)

plt.show()

Cats are more often classified as dogs than the other way around. Let's look at some examples where the model made mistakes:

In [ ]:
errors = (Y_pred_classes != test_gen.classes)
errors_example = np.argsort(errors)[-10:]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 10))
for i, idx in enumerate(errors_example, 1):
    plt.subplot(2, 5, i)
    plt.imshow(X_test[idx])
    plt.title(f"True: {(class_names[Y_true_classes[idx]-1])}\nPred: {(class_names[Y_pred_classes[idx]-1])}\n{np.max(Y_pred[idx]):.2f}")
    plt.axis('off')
plt.tight_layout()
plt.show()

There are also some errors in the dataset itself: some images of cats are labeled as dogs. This affects the quality of the model.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Model

print("Создаем модель для дообучения:")

base_model = model.layers[0]

base_model.trainable = False

x = Dense(256, activation='relu')(base_model.output)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(2, activation='softmax')(x)

hybrid_model = Model(inputs=base_model.input, outputs=predictions)

hybrid_model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    metrics=['accuracy']
)

hybrid_model.summary()

In [ ]:
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

def setup_regularization(model):
    """
    Настраивает регуляризацию для существующей модели
    """

    for layer in model.layers:
        if hasattr(layer, 'layers'):
            for base_layer in layer.layers[:-10]:
                base_layer.trainable = False
        elif not isinstance(layer, layers.Dense):
            layer.trainable = False


    model.compile(
        optimizer=Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

regularized_model = setup_regularization(hybrid_model)

In [ ]:
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

callbacks = [
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'best_regularized_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

regularized_model.compile(
    optimizer=Adam(learning_rate=1e-5, beta_1=0.9, beta_2=0.999),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_regularized = regularized_model.fit(
    train_gen,
    epochs=3,
    validation_data=val_gen,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

After 3 epochs, the model reached an accuracy of 0.95 (95%).

Let's evaluate the improved model and check the plots to see whether it is overfitting:

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(history_regularized.history['val_accuracy'], '-o', label='validation accuracy')
ax1.plot(history_regularized.history['accuracy'], '--s', label='training accuracy')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history_regularized.history['val_loss'], '-o', label='validation loss')
ax2.plot(history_regularized.history['loss'], '--s', label='training loss')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Training accuracy:", [f"{acc:.4f}" for acc in history_regularized.history['accuracy']])
print("Validation accuracy:", [f"{acc:.4f}" for acc in history_regularized.history['val_accuracy']])
print("Final training accuracy:", f"{history_regularized.history['accuracy'][-1]:.4f}")
print("Final validation accuracy:", f"{history_regularized.history['val_accuracy'][-1]:.4f}")

The model does not appear to be overfitting. Accuracy improves on both the training and validation data as the number of epochs increases.

In [ ]:
x_batch, y_batch = next(test_gen)
predictions = hybrid_model.predict(x_batch)

class_names = ['cats', 'dogs']

plt.figure(figsize=(15, 10))
for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(x_batch[i])

    true_class_idx = np.argmax(y_batch[i])
    pred_class_idx = np.argmax(predictions[i])
    confidence = np.max(predictions[i])

    true_class = class_names[true_class_idx]
    pred_class = class_names[pred_class_idx]

    color = 'green' if true_class_idx == pred_class_idx else 'red'

    plt.title(f"true: {true_class}\npred: {pred_class}\nconf: {confidence:.2%}",
              color=color, fontsize=10)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("Проверяем распределение классов в тестовом датасете:")

test_gen.reset()
class_counts = {0: 0, 1: 0}

for i in range(len(test_gen)):
    x_batch, y_batch = test_gen[i]
    for y in y_batch:
        class_idx = np.argmax(y)
        class_counts[class_idx] += 1
    if i >= len(test_gen) - 1:
        break

print(f"Количество кошек (0): {class_counts[0]}")
print(f"Количество собак (1): {class_counts[1]}")
print(f"Всего изображений: {sum(class_counts.values())}")

In [ ]:
print("Ищем больше изображений собак:")

test_gen.reset()

dog_images = []
dog_labels = []

for i in range(len(test_gen)):
    x_batch, y_batch = test_gen[i]

    for j in range(len(y_batch)):
        if np.argmax(y_batch[j]) == 1:  # dogs = 1
            dog_images.append(x_batch[j])
            dog_labels.append(y_batch[j])

    if i % 10 == 0:
        print(f"Проверено батчей: {i+1}/{len(test_gen)}, найдено собак: {len(dog_images)}")

    if len(dog_images) >= 20:
        break

print(f"Найдено {len(dog_images)} изображений собак")

if dog_images:
    dog_array = np.array(dog_images)
    dog_predictions = hybrid_model.predict(dog_array)

    num_to_show = min(12, len(dog_images))

    fig = plt.figure(figsize=(20, 15))

    rows = 4
    cols = 3

    for i in range(num_to_show):
        plt.subplot(rows, cols, i+1)
        plt.imshow(dog_images[i])

        true_class_idx = np.argmax(dog_labels[i])
        pred_class_idx = np.argmax(dog_predictions[i])
        confidence = np.max(dog_predictions[i])

        true_class = "Собака"
        pred_class = "Собака" if pred_class_idx == 1 else "Кошка"

        color = 'green' if true_class_idx == pred_class_idx else 'red'

        plt.title(f"Собака {i+1}\nПредсказание: {pred_class}\nУверенность: {confidence:.2%}",
                  color=color, fontsize=11)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

    correct_dogs = sum(1 for i in range(len(dog_predictions)) if np.argmax(dog_predictions[i]) == 1)
    print(f"\nСтатистика по всем {len(dog_predictions)} собакам:")
    print(f"Собаки определены правильно: {correct_dogs}/{len(dog_predictions)} ({correct_dogs/len(dog_predictions):.1%})")
    print(f"Ошибок: {len(dog_predictions) - correct_dogs}")

# Building the Application

## Application Features

- The user uploads an image
- The model classifies the uploaded image
- The prediction confidence is displayed
- A separate bar chart shows the probabilities for all classes, not only the top prediction

In [ ]:
%%capture
!pip install streamlit pyngrok

In [ ]:
print("Название дообученной модели:", hybrid_model.name)

In [ ]:
hybrid_model.save('functional_8.keras')

In [ ]:
# создание файла с приложением в колабе
%%writefile streamlit_ml_app.py

import streamlit as st
import numpy as np
from PIL import Image, ImageOps
import tensorflow as tf
import pandas as pd
import plotly.express as px

st.set_page_config(
    page_title="Классификатор Кошек и Собак",
    layout="wide"
)

st.title("Классификатор Кошек и Собак")
st.markdown("Загрузите изображение кошки или собаки для классификации")

CLASS_NAMES = ['кошка', 'собака']
model = tf.keras.models.load_model("functional_8.keras")


def preprocess_image(img, target_size=(224, 224)):
    # Изменение размера
    img = img.resize(target_size)
    # Конвертация в RGB
    if img.mode != 'RGB':
        img = img.convert('RGB')
    # Конвертация в numpy array и нормализация
    img_array = np.array(img) / 255.0
    # Добавление batch dimension
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

# загрузка изображений
uploaded_file = st.file_uploader("Загрузите изображение", type=["jpg", "jpeg", "png"])

if uploaded_file:
    image = Image.open(uploaded_file)
    st.image(image, caption="Загруженное изображение", use_container_width=True)

    # применение модели для классификации
    with st.spinner("Предсказание..."):
        processed = preprocess_image(image)
        predictions = model.predict(processed)[0]
        top_idx = np.argmax(predictions)
        top_class = CLASS_NAMES[top_idx]
        top_conf = predictions[top_idx]

    st.markdown(f"На изображении {top_class} с вероятностью {top_conf:.2%}")

# визуализация вероятностей других классов

st.subheader("Вероятности по классам:")
for i, (class_name, prob) in enumerate(zip(CLASS_NAMES, predictions)):
    st.write(f"{class_name}: {prob:.2%}")

In [ ]:
from pyngrok import ngrok, conf
from google.colab import userdata
from time import sleep

conf.get_default().auth_token = userdata.get('ngrok')
ngrok.kill()  # так как одновременно можно запустить только 3 сессии
sleep(2)  # так как отключение запущенных сессий может происходить не мгновенно

public_url = ngrok.connect(addr=8501, proto="http")
print("Приложение доступно по адресу:", public_url)

!streamlit run streamlit_ml_app.py &> /dev/null &